# Machine Learning for Anti-Money Laundering

***
**Master's thesis: empirical part**

**Notebook 3: Models**

Benchmark: IBM AMLworld, variants HI-Small and LI-Small (Altman et al., 2023)
***

| Notebook | Content | Chapter 4 |
| --- | --- | --- |
| 01 Data and EDA | acquisition, integrity checks, temporal partition, EDA | 4.2–4.5 |
| 02 Features and feature selection | Sets A, B, C; causality verification; screening | 4.6 |
| **03 Models** (this notebook) | one section per family: search, fit, threshold, evaluation | 4.7–4.8 |
| 04 Results | comparison across families, sets and variants | 4.9 |

## Preliminaries

This notebook fits the five model families of Section 4.8 on the three feature sets of both
variants, fixes the decision thresholds on the validation block, and scores the test block
once per seed.

**Requirements:** Notebook 02 must have run for both variants in the same Google account. CPU
runtime with the default memory.

**How to run it.** The full sweep takes several hours, so the notebook works in two steps:

1. With `RUN_SWEEP = False` (the default), *Runtime → Run all* loads the data, runs the tests,
   times one fit per family and projects the total running time (Section 3), then stops. No
   score is computed in this step.
2. Set `RUN_SWEEP = True` and run all again. Every candidate and every seed is written to Drive
   as soon as it finishes; if Colab disconnects, run all again and the sweep continues where it
   stopped. To split the work over sessions, list only some families in `FAMILIES`.

In [ ]:
# Parameters
VARIANTS  = ["HI-Small", "LI-Small"]
FAMILIES  = ["logreg", "svm", "rf", "hgb", "ffnn"]   # families whose sweep runs in this session
RUN_SWEEP = False   # False: stop after the timing run in Section 3; True: run the sweep
DRY_RUN   = False   # True: synthetic data in a separate folder (Notebooks 01 and 02 dry runs first)

In [ ]:
# Import relevant Python packages
import os, sys, json, gc, platform, subprocess, datetime, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")      # quieter TensorFlow
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import set_option
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import confusion_matrix, precision_recall_curve

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # iterations are capped by design
set_option("display.precision", 4)
set_option("display.width", 140)
set_option("display.max_columns", 30)
sns.set_theme(context="notebook", style="whitegrid", palette="viridis")

In [ ]:
# Google Drive, project folder and a record of the computational environment
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
except ImportError:                      # outside Colab: work next to the notebook
    BASE = Path.cwd()
PROJECT = BASE / ("aml-thesis-dryrun" if DRY_RUN else "aml-thesis")
assert (PROJECT / "aml" / "build_features.py").exists(), \
    f"{PROJECT} has no Notebook 02 code: run Notebooks 01 and 02 first (same account, same DRY_RUN)"
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
missing = [v for v in VARIANTS if not (PROJECT / "data" / "features" / f"{v}.parquet").exists()]
assert not missing, f"feature matrices missing for {missing}: run Notebook 02 for them first"

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
header = (f"# recorded {datetime.datetime.now().isoformat(timespec='seconds')} (Notebook 03)\n"
          f"# python {platform.python_version()} | cpus {os.cpu_count()} | ram_gb {ram_gb:.1f}\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(PROJECT / "results" / "environment_nb03.txt").write_text(header + freeze)
print(f"project: {PROJECT}")
print(header)

### Project code

Three modules are added to the `aml` package: the metrics and thresholds (`evaluation`), the
five families and their preprocessing (`models`), and the resumable search and evaluation
loop (`sweep`). Modules from Notebooks 01 and 02 are used unchanged.

In [ ]:
%%writefile aml/evaluation.py
"""Thresholds and metrics (Sections 4.7 and 4.9).

Both thresholds are fixed on the validation block and then applied unchanged to the test
block: the F1-maximising threshold, and the score above which the highest-scoring
ALERT_BUDGET share of validation transactions lies.
"""
import numpy as np
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score


def threshold_f1(y, s) -> float:
    prec, rec, thr = precision_recall_curve(y, s)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return float(thr[int(np.argmax(f1[:-1]))])


def threshold_budget(s, budget: float) -> float:
    """Score of the k-th highest transaction, k = ceil(budget * n)."""
    k = max(1, int(np.ceil(budget * len(s))))
    return float(np.partition(s, len(s) - k)[len(s) - k])


def at_threshold(y, s, tau) -> dict:
    flag = s >= tau
    tp = int((flag & (y == 1)).sum())
    fp = int((flag & (y == 0)).sum())
    fn = int((~flag & (y == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall,
            "f1": f1, "alert_rate": float(flag.mean())}


def top_k(y, s, frac: float, seed: int = 0) -> dict:
    """The highest-scoring ceil(frac * n) transactions of this block, ties broken at random
    with a fixed seed: a rank-based operating point unaffected by tied scores."""
    n = len(s)
    k = max(1, int(np.ceil(frac * n)))
    order = np.lexsort((np.random.default_rng(seed).random(n), -np.asarray(s)))
    tp = int(np.asarray(y)[order[:k]].sum())
    pos = int(np.asarray(y).sum())
    return {"k": k, "tp": tp, "recall": tp / pos if pos else 0.0, "precision": tp / k}


def evaluate(y, s, tau_f1, tau_budget, default=None, budget=None) -> dict:
    y = np.asarray(y)
    both = 0 < y.sum() < len(y)
    out = {"n": int(len(y)), "n_pos": int(y.sum()),
           "ap": float(average_precision_score(y, s)) if both else None,
           "roc_auc": float(roc_auc_score(y, s)) if both else None,
           "at_f1": at_threshold(y, s, tau_f1),
           "at_budget": at_threshold(y, s, tau_budget)}
    if default is not None:
        out["at_default"] = at_threshold(y, s, default)
    if budget is not None and len(y):
        out["top_k"] = top_k(y, s, budget)
    return out


In [ ]:
%%writefile aml/models.py
"""The five model families of Section 4.8 and their preprocessing.

Linear models and the network receive log(1 + x) of the heavy-tailed non-negative features
followed by standardisation, both estimated on the rows they are fitted on (never on
validation or test rows). Tree ensembles receive the features unchanged. Every family is
weighted inversely to class frequency; the network through sample weights.
"""
import os

import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import average_precision_score
from sklearn.utils.fixes import parse_version

SCALED = {"logreg", "svm", "ffnn"}
DEFAULT_THRESHOLD = {"logreg": 0.5, "svm": 0.0, "rf": 0.5, "hgb": 0.5, "ffnn": 0.5}
LOG_EXACT = {"amount_paid_base", "amount_received_base"}
LOG_SUFFIXES = ("_cnt", "_sum", "_n_cp", "_hours_since_prev", "_in_deg", "_out_deg",
                "_two_hop", "_pagerank", "_component_size")
_SK18 = parse_version(sklearn.__version__) >= parse_version("1.8")


def log_columns(names) -> list:
    return [i for i, c in enumerate(names) if c in LOG_EXACT or c.endswith(LOG_SUFFIXES)]


class Preprocessor:
    """log(1 + x) on heavy-tailed columns, then standardisation; fitted on training rows."""

    def __init__(self, names):
        self.names, self.idx = list(names), log_columns(names)

    def _log(self, X):
        Z = np.array(X, dtype=np.float32, copy=True)
        if self.idx:
            Z[:, self.idx] = np.log1p(np.maximum(Z[:, self.idx], 0.0))
        return Z

    def fit(self, X):
        Z = self._log(X)
        self.mean_ = Z.mean(axis=0, dtype=np.float64).astype(np.float32)
        sd = Z.std(axis=0, dtype=np.float64)
        self.scale_ = np.where(sd > 0, sd, 1.0).astype(np.float32)
        return self

    def transform(self, X):
        Z = self._log(X)
        Z -= self.mean_
        Z /= self.scale_
        return Z


class KerasMLP:
    """Feed-forward network of one or two hidden layers (Keras), with balanced sample
    weights and early stopping on validation average precision."""

    def __init__(self, hidden_layers=1, units=64, dropout=0.0, learning_rate=1e-3,
                 batch_size=2048, max_epochs=20, patience=3, class_weight="balanced",
                 random_state=0):
        self.hidden_layers, self.units, self.dropout = int(hidden_layers), int(units), float(dropout)
        self.learning_rate, self.batch_size = float(learning_rate), int(batch_size)
        self.max_epochs, self.patience = int(max_epochs), int(patience)
        self.class_weight, self.random_state = class_weight, int(random_state)

    def fit(self, X, y, X_val=None, y_val=None):
        os.environ.setdefault("KERAS_BACKEND", "tensorflow")
        import keras
        keras.utils.set_random_seed(self.random_state)
        layers = [keras.Input(shape=(X.shape[1],))]
        for _ in range(self.hidden_layers):
            layers.append(keras.layers.Dense(self.units, activation="relu",
                                             kernel_initializer="he_uniform"))
            if self.dropout > 0:
                layers.append(keras.layers.Dropout(self.dropout))
        layers.append(keras.layers.Dense(1, activation="sigmoid"))
        model = keras.Sequential(layers)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
                      loss="binary_crossentropy")
        y = np.asarray(y, dtype=np.float32)
        sw = None
        if self.class_weight == "balanced":
            n, n_pos = len(y), float(y.sum())
            sw = np.where(y == 1, n / (2 * n_pos), n / (2 * (n - n_pos))).astype(np.float32)
        best_ap, best_w, wait = -1.0, None, 0
        for epoch in range(self.max_epochs):
            model.fit(X, y, sample_weight=sw, batch_size=self.batch_size, epochs=1,
                      verbose=0, shuffle=True)
            self.epochs_ = epoch + 1
            if X_val is None:
                continue
            ap = average_precision_score(y_val, model.predict(X_val, batch_size=65536,
                                                              verbose=0).ravel())
            if ap > best_ap:
                best_ap, best_w, wait = ap, model.get_weights(), 0
            else:
                wait += 1
                if wait >= self.patience:
                    break
        if best_w is not None:
            model.set_weights(best_w)
        self.model_, self.best_val_ap_ = model, best_ap
        return self

    def decision_function(self, X):
        return self.model_.predict(X, batch_size=65536, verbose=0).ravel()


def make(family: str, params: dict, seed: int, class_weight="balanced"):
    p = dict(params)
    if family == "logreg":                  # penalised logistic regression, fitted by SGD
        return SGDClassifier(loss="log_loss", penalty="elasticnet", class_weight=class_weight,
                             random_state=seed, **p)
    if family == "svm":
        return SGDClassifier(loss="hinge", penalty="elasticnet", class_weight=class_weight,
                             random_state=seed, **p)
    if family == "rf":
        return RandomForestClassifier(class_weight=class_weight, n_jobs=-1,
                                      random_state=seed, **p)
    if family == "hgb":
        return HistGradientBoostingClassifier(class_weight=class_weight, early_stopping=False,
                                              random_state=seed, **p)
    if family == "ffnn":
        return KerasMLP(class_weight=class_weight, random_state=seed, **p)
    raise ValueError(f"unknown family {family!r}")


def fit(model, X, y, X_val=None, y_val=None):
    if isinstance(model, KerasMLP):
        return model.fit(X, y, X_val, y_val)
    return model.fit(X, y)


def scores(model, X) -> np.ndarray:
    """Continuous scores: probabilities where the model has them, margins for the SVM."""
    if isinstance(model, KerasMLP) or (isinstance(model, SGDClassifier) and model.loss == "hinge"):
        return np.asarray(model.decision_function(X), dtype=np.float64)
    return model.predict_proba(X)[:, 1].astype(np.float64)


def extras(model, names) -> dict:
    """Interpretable by-products, kept for Notebook 04."""
    if isinstance(model, (LogisticRegression, SGDClassifier)):
        coef = model.coef_.ravel()
        return {"coefficients": dict(zip(names, map(float, coef))),
                "n_zero_coefficients": int((coef == 0).sum()),
                "n_iter": int(np.max(model.n_iter_))}
    if isinstance(model, RandomForestClassifier):
        return {"impurity_importance": dict(zip(names, map(float, model.feature_importances_)))}
    if isinstance(model, KerasMLP):
        return {"epochs": model.epochs_, "best_val_ap_during_training": float(model.best_val_ap_)}
    return {}


In [ ]:
%%writefile aml/sweep.py
"""Hyperparameter search, final fits and evaluation, resumable across sessions.

For each variant, family and feature set: candidates are drawn once from the declared grid
(fixed seed), each is fitted on the search subsample of the training block and scored by
average precision on the validation block (the temporal equivalent of RandomizedSearchCV
with a PredefinedSplit); the best candidate is fitted on the full training block with
N_SEEDS seeds; thresholds are fixed on validation and the test block is scored once per
seed. Every candidate and every seed is appended to a JSON-lines file on Drive as soon as
it completes, so an interrupted session resumes where it stopped.
"""
import json
import os
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
from sklearn.model_selection import ParameterSampler

from . import config, models
from . import evaluation as ev
from . import model_params as MP

FEATURES_DIR = config.ROOT / "data" / "features"
STORE_DIR = config.RESULTS / "models"


def _json(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(type(o))


class Store:
    """Append-only JSON-lines records: search candidates, final fits, supplement."""

    def __init__(self, directory=STORE_DIR):
        self.dir = Path(directory)
        (self.dir / "scores").mkdir(parents=True, exist_ok=True)
        self.records = {n: self._load(n) for n in ("search", "final", "supplement")}

    def _load(self, name):
        recs, path = {}, self.dir / f"{name}.jsonl"
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line)
                    recs[r["key"]] = r
                except (json.JSONDecodeError, KeyError):
                    continue          # a line cut off by a crash: that run is repeated
        return recs

    def has(self, name, key):
        return key in self.records[name]

    def add(self, name, rec):
        with open(self.dir / f"{name}.jsonl", "a") as fh:
            fh.write(json.dumps(rec, default=_json) + "\n")
            fh.flush()
            os.fsync(fh.fileno())
        self.records[name][rec["key"]] = rec


@dataclass
class VariantData:
    variant: str
    names: list
    sets: dict
    X_tr: np.ndarray
    y_tr: np.ndarray
    X_va: np.ndarray
    y_va: np.ndarray
    X_te: np.ndarray
    y_te: np.ndarray
    X_tail: np.ndarray
    y_tail: np.ndarray
    search_idx: np.ndarray = field(repr=False, default=None)


def load_variant(variant, features_dir=FEATURES_DIR, results_dir=config.RESULTS,
                 search_fraction=MP.SEARCH_FRACTION, seed=MP.SEARCH_SEED) -> VariantData:
    """Training rows exclude the burn-in day; test rows exclude the tail, which is kept
    apart for a separate report. Set C is loaded once; A and B are column subsets."""
    definition = json.loads((Path(results_dir) / f"feature_sets_{variant}.json").read_text())
    names = definition["sets"]["C"]
    df = pd.read_parquet(Path(features_dir) / f"{variant}.parquet",
                         columns=["block", "is_laundering", "burn_in", "tail"] + names)
    block, y = df["block"].astype(str).to_numpy(), df["is_laundering"].to_numpy(np.int8)
    burn, tail = df["burn_in"].to_numpy(), df["tail"].to_numpy()
    M = df[names].to_numpy(np.float32)
    del df
    masks = {"tr": (block == "train") & ~burn, "va": block == "val",
             "te": (block == "test") & ~tail, "tail": tail}
    parts = {k: (M[m], y[m]) for k, m in masks.items()}
    del M
    n_tr = len(parts["tr"][1])
    rng = np.random.default_rng(seed)
    search_idx = np.sort(rng.choice(n_tr, size=int(round(search_fraction * n_tr)), replace=False))
    sets = {s: [names.index(c) for c in definition["sets"][s]] for s in MP.SETS}
    return VariantData(variant, names, sets, *parts["tr"], *parts["va"], *parts["te"],
                       *parts["tail"], search_idx=search_idx)


def candidates(family, n):
    return list(ParameterSampler(MP.GRIDS[family], n_iter=n, random_state=MP.SEARCH_SEED))


def _key(*parts):
    return "|".join(str(p) for p in parts)


def run_config(store: Store, data: VariantData, family: str, set_name: str,
               n_candidates: int, n_seeds: int, progress=print) -> dict:
    v, idx = data.variant, data.sets[set_name]
    names = [data.names[i] for i in idx]
    cands = candidates(family, n_candidates)
    fixed = MP.FIXED[family]

    # --- search on the training subsample, scored on validation -------------------------
    todo = [i for i in range(len(cands)) if not store.has("search", _key(v, family, set_name, f"cand{i}"))]
    if todo:
        Xs, ys = data.X_tr[data.search_idx][:, idx], data.y_tr[data.search_idx]
        Xv = data.X_va[:, idx]
        if family in models.SCALED:
            pre = models.Preprocessor(names).fit(Xs)
            Xs, Xv = pre.transform(Xs), pre.transform(Xv)
        for i in todo:
            tic = time.perf_counter()
            m = models.make(family, {**fixed, **cands[i]}, seed=0)
            models.fit(m, Xs, ys, Xv, data.y_va)
            ap = float(average_precision_score(data.y_va, models.scores(m, Xv)))
            secs = round(time.perf_counter() - tic, 1)
            store.add("search", {"key": _key(v, family, set_name, f"cand{i}"), "variant": v,
                                 "family": family, "set": set_name, "candidate": i,
                                 "params": cands[i], "val_ap": ap, "seconds": secs})
            progress(f"    {v} {family} set {set_name} candidate {i + 1}/{len(cands)}: "
                     f"val AP {ap:.4f} ({secs} s)")
        del Xs, Xv
    found = [store.records["search"][_key(v, family, set_name, f"cand{i}")] for i in range(len(cands))]
    best = max(found, key=lambda r: r["val_ap"])

    # --- final fits on the full training block -------------------------------------------
    seeds = [s for s in range(n_seeds) if not store.has("final", _key(v, family, set_name, f"seed{s}"))]
    if seeds:
        Xt, Xv, Xte, Xtl = (A[:, idx] for A in (data.X_tr, data.X_va, data.X_te, data.X_tail))
        if family in models.SCALED:
            pre = models.Preprocessor(names).fit(Xt)
            Xt, Xv, Xte, Xtl = (pre.transform(A) for A in (Xt, Xv, Xte, Xtl))
        for s in seeds:
            tic = time.perf_counter()
            m = models.make(family, {**fixed, **best["params"]}, seed=s)
            models.fit(m, Xt, data.y_tr, Xv, data.y_va)
            sv, ste = models.scores(m, Xv), models.scores(m, Xte)
            stl = models.scores(m, Xtl) if len(Xtl) else np.empty(0)
            tau_f1, tau_b = ev.threshold_f1(data.y_va, sv), ev.threshold_budget(sv, MP.ALERT_BUDGET)
            d = models.DEFAULT_THRESHOLD[family]
            rec = {"key": _key(v, family, set_name, f"seed{s}"), "variant": v, "family": family,
                   "set": set_name, "seed": s, "params": best["params"],
                   "search_val_ap": best["val_ap"], "n_features": len(idx),
                   "tau_f1": tau_f1, "tau_budget": tau_b,
                   "val": ev.evaluate(data.y_va, sv, tau_f1, tau_b, d, MP.ALERT_BUDGET),
                   "test": ev.evaluate(data.y_te, ste, tau_f1, tau_b, d, MP.ALERT_BUDGET),
                   "tail": ev.evaluate(data.y_tail, stl, tau_f1, tau_b, d) if len(stl) else None,
                   "seconds": round(time.perf_counter() - tic, 1)}
            if s == 0:
                rec["extras"] = models.extras(m, names)
            out = {"test": ste.astype(np.float32), "tail": stl.astype(np.float32)}
            if s == 0:
                out["val"] = sv.astype(np.float32)
            np.savez_compressed(store.dir / "scores" / f"{v}_{family}_{set_name}_seed{s}.npz", **out)
            store.add("final", rec)
            progress(f"    {v} {family} set {set_name} seed {s}: test AP {rec['test']['ap']:.4f}, "
                     f"recall at budget {rec['test']['at_budget']['recall']:.3f} ({rec['seconds']} s)")
    return best


def run_family(store, data_by_variant: dict, family: str, sets=MP.SETS,
               n_candidates=None, n_seeds=MP.N_SEEDS, progress=print):
    n_candidates = n_candidates or MP.N_CANDIDATES[family]
    for v, data in data_by_variant.items():
        for s in sets:
            progress(f"  {v}, {family}, set {s}")
            run_config(store, data, family, s, n_candidates, n_seeds, progress)
    return summary(store, family=family)


def summary(store, family=None) -> pd.DataFrame:
    rows = []
    for r in store.records["final"].values():
        if family and r["family"] != family:
            continue
        t = r["test"]
        rows.append({"variant": r["variant"], "family": r["family"], "set": r["set"], "seed": r["seed"],
                     "val AP (search)": r["search_val_ap"], "test AP": t["ap"],
                     "recall @ budget": t["at_budget"]["recall"],
                     "precision @ budget": t["at_budget"]["precision"],
                     "alert rate @ budget": t["at_budget"]["alert_rate"],
                     "recall @ top 0.1%": t["top_k"]["recall"],
                     "precision @ F1": t["at_f1"]["precision"], "recall @ F1": t["at_f1"]["recall"],
                     "F1": t["at_f1"]["f1"]})
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    keys = ["variant", "family", "set"]
    metrics = [c for c in df.columns if c not in keys + ["seed"]]
    agg = df.groupby(keys)[metrics].agg(["mean", "std"])
    agg[("seeds", "n")] = df.groupby(keys)["seed"].count()
    return agg


def timing_run(data: VariantData, families, set_name="C", progress=print) -> pd.DataFrame:
    """One fit per family on the search subsample (first declared candidate). Measures
    time only; no score is computed or shown."""
    rows = []
    idx = data.sets[set_name]
    names = [data.names[i] for i in idx]
    Xs, ys = data.X_tr[data.search_idx][:, idx], data.y_tr[data.search_idx]
    Xv = data.X_va[:, idx]
    Xs_scaled = Xv_scaled = None
    for f in families:
        if f in models.SCALED and Xs_scaled is None:
            pre = models.Preprocessor(names).fit(Xs)
            Xs_scaled, Xv_scaled = pre.transform(Xs), pre.transform(Xv)
        X1, X2 = (Xs_scaled, Xv_scaled) if f in models.SCALED else (Xs, Xv)
        tic = time.perf_counter()
        m = models.make(f, {**MP.FIXED[f], **candidates(f, 1)[0]}, seed=0)
        models.fit(m, X1, ys, X2, data.y_va)
        secs = time.perf_counter() - tic
        rows.append({"family": f, "seconds_per_search_fit": round(secs, 1)})
        progress(f"  {f}: {secs:.1f} s")
    return pd.DataFrame(rows).set_index("family")


def projection(timing: pd.DataFrame, n_train: dict, n_features: dict, ref_variant: str,
               n_candidates: dict, n_seeds: int, fraction: float) -> pd.DataFrame:
    """Projected hours, assuming time proportional to rows and to features."""
    rows = []
    for f, t in timing["seconds_per_search_fit"].items():
        total = 0.0
        for v in n_train:
            for s, k in n_features[v].items():
                scale = (n_train[v] / n_train[ref_variant]) * (k / n_features[ref_variant]["C"])
                total += t * scale * (n_candidates[f] + n_seeds / fraction)
        rows.append({"family": f, "candidates": n_candidates[f], "projected_hours": total / 3600})
    out = pd.DataFrame(rows).set_index("family")
    out.loc["total"] = [out["candidates"].sum(), out["projected_hours"].sum()]
    return out.round(2)


def best_configuration(store, variant) -> dict:
    """Highest mean validation AP over complete final fits (validation, never test)."""
    df = pd.DataFrame([{"family": r["family"], "set": r["set"], "val_ap": r["val"]["ap"]}
                       for r in store.records["final"].values() if r["variant"] == variant])
    g = df.groupby(["family", "set"])["val_ap"].agg(["mean", "count"])
    g = g[g["count"] == g["count"].max()]
    family, set_name = g["mean"].idxmax()
    return {"family": family, "set": set_name, "mean_val_ap": float(g["mean"].max())}


def run_supplement(store, data: VariantData, n_seeds=MP.N_SEEDS, progress=print) -> pd.DataFrame:
    """Section 5.7: the best configuration without class weighting, with weighting alone
    (default threshold), and with weighting and the tuned threshold (the main result)."""
    best = best_configuration(store, data.variant)
    family, set_name = best["family"], best["set"]
    params = store.records["final"][_key(data.variant, family, set_name, "seed0")]["params"]
    idx = data.sets[set_name]
    names = [data.names[i] for i in idx]
    Xt, Xv, Xte = (A[:, idx] for A in (data.X_tr, data.X_va, data.X_te))
    if family in models.SCALED:
        pre = models.Preprocessor(names).fit(Xt)
        Xt, Xv, Xte = (pre.transform(A) for A in (Xt, Xv, Xte))
    d = models.DEFAULT_THRESHOLD[family]
    for cond, cw in (("unweighted, default threshold", None), ("weighted, default threshold", "balanced")):
        for s in range(n_seeds):
            key = _key(data.variant, family, set_name, cond, f"seed{s}")
            if store.has("supplement", key):
                continue
            m = models.make(family, {**MP.FIXED[family], **params}, seed=s, class_weight=cw)
            models.fit(m, Xt, data.y_tr, Xv, data.y_va)
            ste = models.scores(m, Xte)
            res = ev.evaluate(data.y_te, ste, d, d)
            store.add("supplement", {"key": key, "variant": data.variant, "family": family,
                                     "set": set_name, "condition": cond, "seed": s,
                                     "test_ap": res["ap"], **res["at_f1"]})
            progress(f"    {cond}, seed {s}: F1 {res['at_f1']['f1']:.3f}")
    rows = [r for r in store.records["supplement"].values() if r["variant"] == data.variant]
    for r in store.records["final"].values():
        if (r["variant"], r["family"], r["set"]) == (data.variant, family, set_name):
            rows.append({"condition": "weighted, tuned threshold (main result)", "seed": r["seed"],
                         "test_ap": r["test"]["ap"], **r["test"]["at_f1"]})
    df = pd.DataFrame(rows)
    cols = ["test_ap", "precision", "recall", "f1", "alert_rate"]
    out = df.groupby("condition")[cols].agg(["mean", "std"])
    out.attrs["configuration"] = best
    return out


In [ ]:
%%writefile tests/test_models.py
import importlib.util
import json

import numpy as np
import pandas as pd
import pytest

from aml import evaluation as ev
from aml import models, sweep


def test_thresholds_and_metrics():
    y = np.array([0, 0, 1, 0, 1, 1, 0, 0, 0, 0])
    s = np.array([.1, .2, .9, .3, .8, .4, .35, .05, .15, .25])
    tau = ev.threshold_f1(y, s)
    m = ev.at_threshold(y, s, tau)
    assert m["f1"] == pytest.approx(max(ev.at_threshold(y, s, t)["f1"] for t in s))
    tb = ev.threshold_budget(s, 0.2)                     # top 2 of 10
    assert tb == 0.8 and ev.at_threshold(y, s, tb)["alert_rate"] == 0.2
    out = ev.evaluate(y, s, tau, tb, default=0.5, budget=0.2)
    assert out["n_pos"] == 3 and out["at_default"]["tp"] == 2
    assert out["top_k"]["k"] == 2 and out["top_k"]["tp"] == 2
    tied = ev.top_k(np.array([1, 0, 0, 0]), np.array([.5, .5, .5, .5]), 0.5)
    assert tied["k"] == 2 and tied["tp"] in (0, 1)        # exactly k flagged despite ties


def test_preprocessor_uses_training_rows_only():
    names = ["amount_paid_base", "hour", "src_out_cnt", "src_in_share"]
    assert models.log_columns(names) == [0, 2]
    tr = np.array([[10, 1, 1, .5], [100, 2, 3, .2], [1000, 3, 7, .9]], dtype=np.float32)
    te = np.array([[1e6, 23, 100, .1]], dtype=np.float32)
    pre = models.Preprocessor(names).fit(tr)
    Z = pre.transform(tr)
    np.testing.assert_allclose(Z.mean(axis=0), 0, atol=1e-6)
    assert pre.transform(te)[0, 0] > 3                  # far outside training range, not rescaled to it


def toy_features(tmp_path, n=6000, seed=0, tail=True):
    rng = np.random.default_rng(seed)
    X = {"amount_paid_base": rng.lognormal(6, 2, n), "hour": rng.integers(0, 24, n).astype(float),
         "src_out_cnt": rng.poisson(3, n).astype(float), "src_in_deg": rng.poisson(2, n).astype(float)}
    logit = -4 + 0.8 * np.log1p(X["src_out_cnt"]) + 0.5 * np.log1p(X["src_in_deg"]) \
        + 0.2 * (np.log1p(X["amount_paid_base"]) - 6)
    y = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(np.int8)
    block = np.where(np.arange(n) < .7 * n, "train", np.where(np.arange(n) < .85 * n, "val", "test"))
    df = pd.DataFrame({"file_row": np.arange(n), "block": pd.Categorical(block, ["train", "val", "test"]),
                       "is_laundering": y, "burn_in": np.arange(n) < .05 * n,
                       "tail": (np.arange(n) >= .99 * n) & tail, "snapshot_minute": 0,
                       **{k: v.astype(np.float32) for k, v in X.items()}})
    (tmp_path / "features").mkdir()
    df.to_parquet(tmp_path / "features" / "HI-Small.parquet", index=False)
    sets = {"A": ["amount_paid_base", "hour"]}
    sets["B"] = sets["A"] + ["src_out_cnt"]
    sets["C"] = sets["B"] + ["src_in_deg"]
    (tmp_path / "feature_sets_HI-Small.json").write_text(json.dumps({"sets": sets}))
    return sweep.load_variant("HI-Small", tmp_path / "features", tmp_path, 0.5, 0)


def test_sweep_resumes_and_summarises(tmp_path):
    data = toy_features(tmp_path)
    assert len(data.y_tail) == 60 and data.X_tr.shape[1] == 4 and data.sets["A"] == [0, 1]
    store = sweep.Store(tmp_path / "models")
    log = []
    for fam in ["logreg", "svm", "rf", "hgb"]:
        sweep.run_family(store, {"HI-Small": data}, fam, sets=("A", "C"),
                         n_candidates=2, n_seeds=2, progress=log.append)
    n_search, n_final = len(store.records["search"]), len(store.records["final"])
    assert (n_search, n_final) == (4 * 2 * 2, 4 * 2 * 2)
    # a second run (as after a crash) fits nothing new
    store2 = sweep.Store(tmp_path / "models")
    sweep.run_family(store2, {"HI-Small": data}, "rf", sets=("A", "C"), n_candidates=2,
                     n_seeds=2, progress=log.append)
    assert len(store2.records["final"]) == n_final
    summ = sweep.summary(store2)
    assert summ.shape[0] == 8 and ("test AP", "mean") in summ.columns
    rec = store2.records["final"]["HI-Small|logreg|C|seed0"]
    assert rec["tail"]["n"] == 60 and "coefficients" in rec["extras"]
    assert (tmp_path / "models" / "scores" / "HI-Small_rf_C_seed1.npz").exists()
    sup = sweep.run_supplement(store2, data, n_seeds=2, progress=log.append)
    assert len(sup) == 3


@pytest.mark.skipif(importlib.util.find_spec("keras") is None, reason="keras not installed")
def test_keras_network_trains_with_early_stopping(tmp_path, monkeypatch):
    data = toy_features(tmp_path)
    store = sweep.Store(tmp_path / "models")
    import aml.model_params as MP
    monkeypatch.setitem(MP.FIXED, "ffnn", {"max_epochs": 3, "patience": 1})
    sweep.run_family(store, {"HI-Small": data}, "ffnn", sets=("C",), n_candidates=1,
                     n_seeds=1, progress=lambda m: None)
    rec = store.records["final"]["HI-Small|ffnn|C|seed0"]
    assert 1 <= rec["extras"]["epochs"] <= 3 and rec["test"]["ap"] is not None


def test_sweep_without_tail_rows(tmp_path):
    data = toy_features(tmp_path, tail=False)
    store = sweep.Store(tmp_path / "models")
    sweep.run_family(store, {"HI-Small": data}, "logreg", sets=("A",), n_candidates=1,
                     n_seeds=1, progress=lambda m: None)
    assert store.records["final"]["HI-Small|logreg|A|seed0"]["tail"] is None


## 1 Design decisions

The cell below fixes every parameter of model training (Sections 4.7 and 4.8). Three points
revise Section 4.8 and were agreed before any model was fitted:

| Specification (Section 4.8) | Here | Reason |
| --- | --- | --- |
| Search on the full training block | search on a uniform 25 % subsample of it; the selected configuration is then fitted on the full block | computation on two CPUs; the class ratio is preserved in expectation, nothing is resampled |
| 60 candidates per family and set | 20 (logistic regression, SVM), 10 (random forest, gradient boosting), 8 (network) | the same; set after the timing run in Section 3, before any score was computed |
| Logistic regression by a batch solver | fitted by stochastic gradient descent with the log loss, like the SVM with the hinge loss | the saga solver needed over two minutes per search fit; both linear families now differ only in their loss |
| Refit on training plus validation before testing | no refit: fit on training, threshold on validation, test scored once | the threshold chosen on validation must belong to the model that is tested |

Unchanged from the specification: class weighting inversely proportional to class frequency
in every family and no resampling; the F1-maximising threshold and the 0.1 % alert budget, both
fixed on validation; average precision as the search criterion and primary measure; five seeds
per final configuration. Because tied scores (common in tree ensembles) can make a threshold
flag more than its share, the realised alert rate is reported next to recall at the budget,
together with recall among the highest-scoring 0.1 % of the test block itself, ties broken at
random with a fixed seed. Linear models and the network receive log(1 + x) of heavy-tailed
features and standardisation, estimated on the rows they are fitted on; tree ensembles receive
the features unchanged. The network stops early when validation average precision stops
improving (validation only, never test). Rows of the burn-in day are excluded from training,
and the tail after 10 September from the primary test evaluation (reported separately).

In [ ]:
%%writefile aml/model_params.py
"""Model training parameters (Sections 4.7-4.9, revised before any model was fitted).

Revisions of Section 4.8, agreed before the sweep: the hyperparameter search is run on a
uniform 25 % subsample of the training block (a computational measure: the class ratio is
preserved in expectation, nothing is resampled), the number of candidates per family is
reduced from sixty, and the selected configuration is fitted on the training block only
(no refit on training plus validation, so the threshold chosen on validation belongs to
the model that is tested). After the timing run (projected 20 hours, before any score was
computed): logistic regression is fitted by stochastic gradient descent like the SVM (the
saga solver needed over two minutes per search fit), the boosting grid is capped at 200
iterations, the network uses a fixed batch of 4,096 and at most 15 epochs, and the boosting
and network searches draw 10 and 8 candidates. Change nothing here once the sweep has started.
"""
from scipy.stats import loguniform

FAMILIES = ("logreg", "svm", "rf", "hgb", "ffnn")
SETS = ("A", "B", "C")
N_CANDIDATES = {"logreg": 20, "svm": 20, "rf": 10, "hgb": 10, "ffnn": 8}
SEARCH_FRACTION = 0.25        # share of training rows used for the search
SEARCH_SEED = 0               # candidate sampling and the search subsample
N_SEEDS = 5                   # final fits per selected configuration
ALERT_BUDGET = 0.001          # highest-scoring 0.1 % of transactions (Section 4.7)

# Declared search spaces (Appendix A). Every candidate uses class_weight="balanced".
GRIDS = {
    "logreg": {"alpha": loguniform(1e-7, 1e-2), "l1_ratio": [0.0, 0.25, 0.5, 0.75, 1.0]},
    "svm": {"alpha": loguniform(1e-7, 1e-2), "l1_ratio": [0.0, 0.15, 0.5, 0.85, 1.0]},
    "rf": {"n_estimators": [100, 200], "max_depth": [8, 12, 16, 24],
           "min_samples_leaf": [10, 50, 200], "max_features": ["sqrt", 0.3, 0.5],
           "max_samples": [0.05, 0.1, 0.2]},
    "hgb": {"learning_rate": loguniform(0.03, 0.3), "max_iter": [100, 200],
            "max_leaf_nodes": [15, 31, 63], "min_samples_leaf": [20, 100, 500],
            "l2_regularization": loguniform(1e-3, 10)},
    "ffnn": {"hidden_layers": [1, 2], "units": [32, 64, 128], "dropout": [0.0, 0.2],
             "learning_rate": loguniform(1e-4, 3e-3)},
}
# Settings held fixed for every candidate of a family.
FIXED = {
    "logreg": {"max_iter": 30, "tol": 1e-4},
    "svm": {"max_iter": 30, "tol": 1e-4},
    "rf": {},
    "hgb": {},
    "ffnn": {"max_epochs": 15, "patience": 3, "batch_size": 4096},
}


In [ ]:
# Use the modules just written: discard any copy imported earlier in this session
import importlib
for name in [n for n in list(sys.modules) if n.split(".")[0] in ("aml", "tests")]:
    del sys.modules[name]
importlib.invalidate_caches()

from aml import sweep, models
from aml import model_params as MP
N_CAND, N_SEEDS = dict(MP.N_CANDIDATES), MP.N_SEEDS
if DRY_RUN:   # a quick mechanical check, not an experiment
    N_CAND, N_SEEDS = {f: 2 for f in MP.FAMILIES}, 2
    MP.FIXED["ffnn"] = {"max_epochs": 2, "patience": 1}
    RUN_SWEEP = True

# Declared search spaces (Appendix A)
def describe(v):
    if hasattr(v, "dist"):
        a, b = v.support()
        return f"{v.dist.name}({a:g}, {b:g})"
    return ", ".join(map(str, v))
display(pd.DataFrame([{"family": f, "hyperparameter": k, "values": describe(v), "candidates": N_CAND[f],
                       "fixed": ", ".join(f"{a}={b}" for a, b in MP.FIXED[f].items())}
                      for f in MP.FAMILIES for k, v in MP.GRIDS[f].items()]).set_index(["family", "hyperparameter"]))

In [ ]:
# Run the test suite (all notebooks; synthetic data only)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                     capture_output=True, text=True)
print(res.stdout[-2500:], res.stderr[-2000:])
assert res.returncode == 0, "tests failed"

## 2 Data

For each variant, Set C is loaded once from Notebook 02's matrix; Sets A and B are column
subsets of it. Training rows exclude the burn-in day, test rows exclude the tail, and the
search subsample is drawn once per variant, so every family and set is searched on the same rows.

In [ ]:
# Data: load both variants and summarise the partitions used for modelling
DATA = {v: sweep.load_variant(v, search_fraction=MP.SEARCH_FRACTION, seed=MP.SEARCH_SEED) for v in VARIANTS}
store = sweep.Store()
display(pd.DataFrame({v: {"training rows (after burn-in)": len(d.y_tr), "training laundering": int(d.y_tr.sum()),
                          "search subsample rows": len(d.search_idx),
                          "search subsample laundering": int(d.y_tr[d.search_idx].sum()),
                          "validation rows": len(d.y_va), "validation laundering": int(d.y_va.sum()),
                          "test rows (without tail)": len(d.y_te), "test laundering": int(d.y_te.sum()),
                          "tail rows": len(d.y_tail), "tail laundering": int(d.y_tail.sum()),
                          **{f"features, set {s}": len(i) for s, i in d.sets.items()}}
                      for v, d in DATA.items()}).astype(object))
FIG, TAB = PROJECT / "results" / "figures", PROJECT / "results" / "tables"

In [ ]:
# Helpers for the per-family sections: summary table, PR curves, confusion matrix
SET_COLOURS = {"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}

def table(family):
    s = sweep.summary(store, family=family)
    if s.empty:
        print(f"{family}: no final results yet")
        return s
    keep = ["test AP", "recall @ budget", "alert rate @ budget", "recall @ top 0.1%",
            "precision @ F1", "recall @ F1", "F1"]
    out = pd.DataFrame({m: s[(m, "mean")].map("{:.4f}".format) + " ± " + s[(m, "std")].fillna(0).map("{:.4f}".format)
                        for m in keep})
    out["val AP (search)"] = s[("val AP (search)", "mean")].round(4)
    out["seeds"] = s[("seeds", "n")]
    display(out)
    out.to_csv(TAB / f"models_{family}.csv")
    return s

def scores(variant, family, set_name, seed=0):
    path = store.dir / "scores" / f"{variant}_{family}_{set_name}_seed{seed}.npz"
    return np.load(path) if path.exists() else None

def pr_curves(family):
    fig, axes = plt.subplots(1, len(DATA), figsize=(7 * len(DATA), 5), squeeze=False)
    for ax, (v, d) in zip(axes[0], DATA.items()):
        for s in MP.SETS:
            sc = scores(v, family, s)
            if sc is None:
                continue
            p, r, _ = precision_recall_curve(d.y_te, sc["test"])
            ap = store.records["final"][f"{v}|{family}|{s}|seed0"]["test"]["ap"]
            ax.plot(r, p, color=SET_COLOURS[s], label=f"set {s} (AP {ap:.3f})")
        ax.axhline(d.y_te.mean(), ls=":", color="grey", label="base rate")
        ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title(f"{v}: {family}, test block, seed 0")
        ax.legend()
    fig.tight_layout()
    fig.savefig(FIG / f"pr_{family}.png", dpi=200, bbox_inches="tight")
    plt.show()

def confusion(family, variant="HI-Small", set_name="C"):
    sc = scores(variant, family, set_name)
    if sc is None:
        return
    rec = store.records["final"][f"{variant}|{family}|{set_name}|seed0"]
    cm = confusion_matrix(DATA[variant].y_te, (sc["test"] >= rec["tau_f1"]).astype(int))
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="viridis", cbar=False, ax=ax,
                xticklabels=["licit", "laundering"], yticklabels=["licit", "laundering"])
    ax.set_xlabel("predicted"); ax.set_ylabel("actual")
    ax.set_title(f"{variant}, {family}, set {set_name}: test block at the F1 threshold")
    fig.savefig(FIG / f"confusion_{family}_{variant}_{set_name}.png", dpi=200, bbox_inches="tight")
    plt.show()

def run(family):
    if family in FAMILIES and RUN_SWEEP:
        sweep.run_family(store, DATA, family, n_candidates=N_CAND[family], n_seeds=N_SEEDS)
    elif family not in FAMILIES:
        print(f"{family} is not in FAMILIES for this session; showing stored results")

## 3 Compute budget

One fit per family on the search subsample of HI-Small, Set C (the first declared candidate),
timed but not scored. The projection assumes time proportional to rows and features; it is
approximate, particularly for the network, whose early stopping varies. If the total is too long
for the time available, reduce `N_CANDIDATES` in Section 1 now, before the sweep starts: at this
point no validation or test score has been seen, so the change is a declared budget, not a
choice informed by results.

In [ ]:
# Compute budget: time one search fit per family and project the full sweep
timing_file = store.dir / "timing.json"
if RUN_SWEEP and timing_file.exists():
    print("Timing run already recorded (" + json.loads(timing_file.read_text())["recorded"][:16] + "); skipped.")
else:
    ref = "HI-Small" if "HI-Small" in DATA else VARIANTS[0]
    timing = sweep.timing_run(DATA[ref], MP.FAMILIES)
    n_train = {v: len(d.y_tr) for v, d in DATA.items()}
    n_feat = {v: {s: len(i) for s, i in d.sets.items()} for v, d in DATA.items()}
    proj = sweep.projection(timing, n_train, n_feat, ref, N_CAND, N_SEEDS, MP.SEARCH_FRACTION)
    display(timing.join(proj, how="outer"))
    timing_file.write_text(json.dumps({"timing": timing.to_dict(), "projection": proj.to_dict(),
                                       "recorded": datetime.datetime.now().isoformat()}, indent=2))
if not RUN_SWEEP:
    raise SystemExit("Timing run complete. Check the projection; then set RUN_SWEEP = True and run all again.")

## 4 Logistic regression

Penalised logistic regression with an elastic-net penalty, fitted by stochastic gradient descent
with the log loss: `l1_ratio` = 0 is the ridge penalty, 1 the lasso, values between mix them, and
`alpha` sets its strength. The L1 part performs the embedded feature
selection of Section 4.6; the number of coefficients set exactly to zero is reported below.

In [ ]:
# Logistic regression: search, final fits and evaluation (resumes if interrupted)
run("logreg")
s_logreg = table("logreg")

In [ ]:
# Logistic regression: precision-recall curves on the test block and the confusion matrix
pr_curves("logreg")
confusion("logreg")

In [ ]:
# Logistic regression: embedded selection (coefficients set exactly to zero, seed 0)
rows = [{"variant": r["variant"], "set": r["set"], "features": r["n_features"],
         "zero coefficients": r["extras"]["n_zero_coefficients"], "l1_ratio": r["params"]["l1_ratio"],
         "alpha": r["params"]["alpha"], "iterations": r["extras"]["n_iter"]}
        for r in store.records["final"].values() if r["family"] == "logreg" and r["seed"] == 0]
display(pd.DataFrame(rows).sort_values(["variant", "set"]) if rows else "no results yet")

## 5 Linear support vector machine

A linear SVM fitted by stochastic gradient descent with the hinge loss (kernel methods do not scale
to millions of rows). It produces margins rather than probabilities; average precision and both
thresholds work on margins directly, and no calibration is applied (calibration with random folds
would mix time periods).

In [ ]:
# Linear support vector machine: search, final fits and evaluation (resumes if interrupted)
run("svm")
s_svm = table("svm")

In [ ]:
# Linear support vector machine: precision-recall curves on the test block and the confusion matrix
pr_curves("svm")
confusion("svm")

## 6 Random forest

Bagged decision trees. `max_samples` bounds the bootstrap sample per tree, which keeps fitting on
millions of rows feasible on two CPUs; depth and leaf size are searched. Impurity importances of
the Set C model are shown below as a diagnostic.

In [ ]:
# Random forest: search, final fits and evaluation (resumes if interrupted)
run("rf")
s_rf = table("rf")

In [ ]:
# Random forest: precision-recall curves on the test block and the confusion matrix
pr_curves("rf")
confusion("rf")

In [ ]:
# Random forest: impurity importances of the Set C model (HI-Small, seed 0), diagnostic only
rec = store.records["final"].get("HI-Small|rf|C|seed0")
if rec:
    imp = pd.Series(rec["extras"]["impurity_importance"]).sort_values().tail(25)
    fig, ax = plt.subplots(figsize=(9, 8))
    imp.plot.barh(ax=ax)
    ax.set_xlabel("mean decrease in impurity")
    ax.set_title("Random forest, HI-Small, set C: top 25 impurity importances")
    fig.savefig(FIG / "rf_importance_HI-Small_C.png", dpi=200, bbox_inches="tight")
    plt.show()

## 7 Histogram-based gradient boosting

scikit-learn's histogram-based gradient boosting, algorithmically close to LightGBM (Section 4.8).
Early stopping is disabled, so the number of iterations is a searched hyperparameter like the others.

In [ ]:
# Histogram-based gradient boosting: search, final fits and evaluation (resumes if interrupted)
run("hgb")
s_hgb = table("hgb")

In [ ]:
# Histogram-based gradient boosting: precision-recall curves on the test block and the confusion matrix
pr_curves("hgb")
confusion("hgb")

## 8 Feed-forward neural network

A Keras network of one or two hidden layers (ReLU, He initialisation, optional dropout, Adam,
binary cross-entropy, batches of 4,096), as in Lecture 7b. Class weighting enters as sample weights. Training stops
when validation average precision has not improved for three epochs, and the best weights are kept.

In [ ]:
# Feed-forward neural network: search, final fits and evaluation (resumes if interrupted)
run("ffnn")
s_ffnn = table("ffnn")

In [ ]:
# Feed-forward neural network: precision-recall curves on the test block and the confusion matrix
pr_curves("ffnn")
confusion("ffnn")

## 9 Comparison across families

In [ ]:
# Comparison: test average precision and recall at the alert budget (mean over seeds)
all_s = sweep.summary(store)
if not all_s.empty:
    for metric in ["test AP", "recall @ budget", "recall @ top 0.1%"]:
        piv = all_s[(metric, "mean")].unstack(["variant", "set"]).round(4)
        print(metric)
        display(piv)
        piv.to_csv(TAB / f"models_{metric.replace(' ', '_').replace('@', 'at').replace('%', 'pct')}.csv")
    hi = all_s.loc["HI-Small"] if "HI-Small" in all_s.index.get_level_values(0) else None
    if hi is not None:
        m, sd = hi[("test AP", "mean")].unstack("set"), hi[("test AP", "std")].unstack("set")
        ax = m.plot.bar(yerr=sd, capsize=3, figsize=(10, 5), color=[SET_COLOURS[c] for c in m.columns])
        ax.set_ylabel("test average precision (mean ± sd over seeds)")
        ax.set_title("HI-Small: test average precision by family and feature set")
        plt.savefig(FIG / "models_ap_HI-Small.png", dpi=200, bbox_inches="tight")
        plt.show()

In [ ]:
# The tail after 10 September, reported separately (F1 threshold fixed on validation)
rows = [{"variant": r["variant"], "family": r["family"], "set": r["set"], "seed": r["seed"],
         "tail AP": r["tail"]["ap"], "tail recall": r["tail"]["at_f1"]["recall"],
         "tail precision": r["tail"]["at_f1"]["precision"]}
        for r in store.records["final"].values() if r.get("tail")]
if rows:
    tail = pd.DataFrame(rows).groupby(["variant", "family", "set"])[["tail AP", "tail recall", "tail precision"]].mean()
    display(tail.round(3))
    tail.to_csv(TAB / "models_tail.csv")

## 10 Supplementary comparison of imbalance treatments (Section 5.7)

On the single best configuration of HI-Small, chosen by mean *validation* average precision so
that no test result informs the choice: without class weighting at the default threshold, with
class weighting at the default threshold, and with class weighting and the threshold tuned on
validation (the main result above).

In [ ]:
# Supplement: imbalance treatments on the best configuration (runs once all HI-Small finals exist)
expected = len(MP.FAMILIES) * len(MP.SETS) * N_SEEDS
done = sum(r["variant"] == "HI-Small" for r in store.records["final"].values())
if "HI-Small" in DATA and done >= expected and RUN_SWEEP:
    sup = sweep.run_supplement(store, DATA["HI-Small"], n_seeds=N_SEEDS)
    print("configuration:", sup.attrs["configuration"])
    display(sup.round(4))
    sup.to_csv(TAB / "imbalance_supplement_HI-Small.csv")
else:
    print(f"waiting for all HI-Small final fits ({done} of {expected} done)")

## 11 Findings to record

1. Section 3: the measured time per fit and the projected total (Section 4.10).
2. Sections 4–8: per family and set, test average precision and recall at the alert budget (mean
   ± sd over five seeds). Where intervals overlap, say so rather than ranking (Section 4.9).
3. RQ2: does average precision rise from Set A to B and from B to C, in every family and both
   variants?
4. Section 4: how many coefficients the elastic-net penalty set to zero (embedded selection).
5. Section 9: how the tail behaves compared with the regular test block.
6. Section 10: the effect of class weighting and of the tuned threshold on the best configuration.

## 12 Output to send back

In [ ]:
# Output: where the results are
print("Records:", {k: len(v) for k, v in store.records.items()})
print("Send the results folder as a zip; the model results are in results/models/ "
      "(the scores/ subfolder may be left out if the zip is too large).")